# Phase 3 — Baseline 1: Double-Logistic Curve Fitting

Classical EVI-based phenology baseline. Predicts phases 15, 18, 21, 24 from SOS/PEAK/EOS.

This notebook re-runs the same logic as `baseline_doublelogistic.py` but in a
cell-by-cell, resumable way. Outputs go to `phase3_outputs/dlog/`.


## 1. Setup

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys, os

# Ensure project root is on the path so phase3_baselines and phenology_pipeline import
PROJECT_ROOT = Path('/fs1/home/atikul/Phenology')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
print('cwd:', os.getcwd())
print('python:', sys.version.split()[0])


cwd: /fs1/home/atikul/Phenology/notebooks
python: 3.13.11


In [2]:
TENSORS_DIR = PROJECT_ROOT / 'cache' / 'tensors'
OUTPUT_DIR  = PROJECT_ROOT / 'phase3_outputs' / 'dlog'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

YEARS = [2017, 2018, 2019, 2020, 2021]
print('Tensors dir:', TENSORS_DIR)
print('Output  dir:', OUTPUT_DIR)


Tensors dir: /fs1/home/atikul/Phenology/cache/tensors
Output  dir: /fs1/home/atikul/Phenology/phase3_outputs/dlog


## 2. Load bundles

In [3]:
from phase3_baselines.dataset import load_all_bundles, label_peak_days
bundles = load_all_bundles(TENSORS_DIR, years=YEARS)
for y, b in bundles.items():
    print(f'{y}: N={len(b["xid"])}  T={b["T"]}  F={len(b["feature_names"])}  '
          f'fully-labeled={int(b["label_mask"].all(axis=1).sum())}')


2017: N=5608  T=426  F=19  fully-labeled=3745
2018: N=5392  T=426  F=19  fully-labeled=3605
2019: N=5541  T=426  F=19  fully-labeled=3603
2020: N=5181  T=426  F=19  fully-labeled=3429
2021: N=5093  T=426  F=19  fully-labeled=3357


## 3. Fit double-logistic per record

In [4]:
from phase3_baselines.baseline_doublelogistic import predict_for_bundle

per_year_preds = {}
for y in YEARS:
    if y not in bundles:
        continue
    print(f'\n=== Year {y} ===')
    pred = predict_for_bundle(bundles[y])
    truth = label_peak_days(bundles[y]['Y'], bundles[y]['label_mask']).astype(np.float32)
    per_year_preds[y] = (pred, truth)



=== Year 2017 ===
  curve-fit: 5351/5425 succeeded (98.6%)

=== Year 2018 ===
  curve-fit: 5385/5392 succeeded (99.9%)

=== Year 2019 ===
  curve-fit: 5314/5541 succeeded (95.9%)

=== Year 2020 ===
  curve-fit: 5051/5181 succeeded (97.5%)

=== Year 2021 ===
  curve-fit: 4973/5092 succeeded (97.7%)


## 4. Evaluate

In [5]:
from phase3_baselines.evaluate import evaluate_predictions
from phase3_baselines.resumable import append_row

results_csv = OUTPUT_DIR / 'dlog_results.csv'
if results_csv.exists():
    results_csv.unlink()   # fresh start

rows = []
for y, (pred, truth) in per_year_preds.items():
    b = bundles[y]
    pred_eval = np.where(np.isnan(pred), -1, pred).astype(np.float32)
    report = evaluate_predictions(
        pred_days=pred_eval, true_days=truth,
        phase_ids=b['phase_ids'],
        years=b['harvest_year'], ranks=b['rank'],
    )
    for phase, m in report['per_phase'].iterrows():
        row = {'test_year': int(y), 'phase': int(phase), **m.to_dict()}
        append_row(results_csv, row)
        rows.append(row)

results = pd.DataFrame(rows)
print('\n=== MAE per (phase, year) ===')
print(results.pivot(index='phase', columns='test_year', values='mae').round(2).to_string())
print('\n=== R² per (phase, year) ===')
print(results.pivot(index='phase', columns='test_year', values='r2').round(2).to_string())
print('\n=== within±5d per (phase, year) ===')
print(results.pivot(index='phase', columns='test_year', values='within_5d').round(2).to_string())



=== MAE per (phase, year) ===
test_year   2017   2018   2019   2020   2021
phase                                       
10           NaN    NaN    NaN    NaN    NaN
12           NaN    NaN    NaN    NaN    NaN
15         20.34  13.92  23.44  26.44  25.39
18         20.19  17.53  26.76  19.55  21.84
19           NaN    NaN    NaN    NaN    NaN
21         23.68  25.95  23.44  22.98  24.60
24         29.32  26.39  25.00  25.31  30.94

=== R² per (phase, year) ===
test_year   2017   2018   2019   2020   2021
phase                                       
10           NaN    NaN    NaN    NaN    NaN
12           NaN    NaN    NaN    NaN    NaN
15         -3.05  -2.36  -5.00  -8.81  -5.44
18        -13.91  -8.79 -14.21  -7.12 -11.05
19           NaN    NaN    NaN    NaN    NaN
21         -7.82  -9.10  -6.48  -5.66  -8.05
24         -9.93 -14.72  -9.00 -10.29 -10.21

=== within±5d per (phase, year) ===
test_year  2017  2018  2019  2020  2021
phase                                  
10          

## 5. Save per-record predictions

In [6]:
preds_npz = OUTPUT_DIR / 'dlog_predictions.npz'
save_dict = {}
for y, (pred, truth) in per_year_preds.items():
    b = bundles[y]
    save_dict[f'{y}__xid']   = b['xid']
    save_dict[f'{y}__bag']   = b['bag_id']
    save_dict[f'{y}__rank']  = b['rank']
    save_dict[f'{y}__yt']    = truth
    save_dict[f'{y}__yp']    = pred
np.savez_compressed(preds_npz, **save_dict)
print(f'Saved predictions -> {preds_npz}')


Saved predictions -> /fs1/home/atikul/Phenology/phase3_outputs/dlog/dlog_predictions.npz


---\n**Done.** Move on to `02_rf_baseline.ipynb`.